# 02 — Repair and validate the independent concept finder

G-SWAP has passed, so this notebook may build the mean-difference (MD)
direction family. It reuses the leakage-audited 40-concept cue manifest, then
selects residual pooling and layer by leave-one-training-template-out retrieval
inside the clean-readout-selected Stage-1 workspace. Explicit probe wording is
also selected on training cues only; held-out cues decide G-DIR.

In [1]:
import json
import os
import sys
from pathlib import Path

ROOT = Path('/home/jovyan/j-space-thoughts')
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ['HF_HOME'] = str(Path.home() / '.cache/huggingface')
os.environ['HF_HUB_CACHE'] = str(Path.home() / '.cache/huggingface/hub')
os.environ['HUGGINGFACE_HUB_CACHE'] = str(Path.home() / '.cache/huggingface/hub')

metrics = json.loads((ROOT / 'results/metrics.json').read_text())
repair = metrics['repair_v2']
assert repair['gate_ledger']['g_swap'] == 'PASS'
workspace_layers = repair['stage1']['g_swap']['canonical_configuration']['layers']

from src.jlens_iface import load_published_lens
from src.model_utils import load_model
from src.v2_repair import MODEL_ID

bundle = load_model(MODEL_ID)
lens = load_published_lens(MODEL_ID)
print('workspace layers fixed before MD heldout evaluation:', workspace_layers)

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

workspace layers fixed before MD heldout evaluation: [13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24]


In [2]:
from src.v2_concept import run_stage1c

stage1c = run_stage1c(bundle, lens, workspace_layers=workspace_layers)
r = stage1c['retrieval']['top1_at_fixed_layer']
e = stage1c['explicit_known_answer']
a = stage1c['cosine_alignment']['raw_WU_J']['fixed_layer']
print({
    'G-DIR': stage1c['status'],
    'pooling': stage1c['position_selection']['selected_pooling'],
    'fixed_layer': stage1c['fixed_validation_layer'],
    'retrieval_top1': r['estimate'],
    'retrieval_ci': [r['ci_low'], r['ci_high']],
    'chance': stage1c['chance_retrieval'],
    'explicit_template': e['selected_template'],
    'heldout_exact_top5': e['heldout_top5']['estimate'],
    'cosine_md_raw_jlens': a['estimate'],
})
print('criteria:', stage1c['criteria'])

{'G-DIR': 'PASS', 'pooling': 'cue_last4', 'fixed_layer': 24, 'retrieval_top1': 0.55, 'retrieval_ci': [0.44119507889355963, 0.6542231081528878], 'chance': 0.025, 'explicit_template': 'question_one_word', 'heldout_exact_top5': 0.8875, 'cosine_md_raw_jlens': 0.13192627811804414}
criteria: {'retrieval_estimate_at_least_4x_chance': True, 'retrieval_ci_above_chance': True, 'retrieval_permutation_p_below_0.01': True, 'retrieval_auroc_ci_above_half': True, 'known_answer_top5_at_least_0.80': True, 'heldout_sign_fraction_at_least_0.80': True, 'stability_median_at_least_0.70': True, 'silent_top10_at_most_0.25': True, 'unit_norm_within_1e-5': True}


In [3]:
from src.v2_concept import persist_stage1c

metrics = persist_stage1c(stage1c)
gate = metrics['repair_v2']['gate_ledger']['g_dir']
print('Persisted G-DIR:', gate)
assert gate in {'PASS', 'DROPPED_MD'}
assert metrics['repair_v2']['gate_ledger']['stage3_science'] == 'PROHIBITED'

Persisted G-DIR: PASS


In [4]:
import gc
import torch

del stage1c, metrics, lens, bundle
gc.collect()
torch.cuda.empty_cache()
print('Notebook 02 complete. Next: READ validation; science remains prohibited.')

Notebook 02 complete. Next: READ validation; science remains prohibited.
